Scenario
You work for a healthcare retail chain.
Management wants to understand:
sales performance
customer behavior
store performance
 

In [0]:
# Dataset 1: Customers
 
customers = [
    (1, "Arjun", "Kochi"),
    (2, "Meera", "Chennai"),
    (3, "Rahul", "Bangalore"),
    (4, "Sneha", "Hyderabad"),
    (5, "Vikram", "Delhi"),
    (6, "Asha", "Kochi"),
    (7, "Kiran", "Chennai"),
    (8, "Neha", "Bangalore")
]
 
df_customers = spark.createDataFrame(
    customers,
    ["customer_id","customer_name","city"]
)
 

In [0]:
# Dataset 2: Orders
 
orders = [
    (101,1,"Medicine",2500,"2024-01-05"),
    (102,1,"Supplements",1800,"2024-01-15"),
    (103,2,"Medical Device",7500,"2024-01-20"),
    (104,3,"Medicine",3200,"2024-02-01"),
    (105,4,"Supplements",2200,"2024-02-05"),
    (106,5,"Medical Device",8500,"2024-02-08"),
    (107,6,"Medicine",2900,"2024-02-12"),
    (108,7,"Supplements",2100,"2024-02-18"),
    (109,8,"Medicine",3400,"2024-03-01"),
    (110,2,"Medicine",2800,"2024-03-04"),
    (111,3,"Medical Device",9000,"2024-03-10"),
    (112,4,"Medicine",3100,"2024-03-15"),
    (113,5,"Supplements",2500,"2024-03-20"),
    (114,6,"Medicine",2700,"2024-04-01"),
    (115,7,"Medical Device",8000,"2024-04-05"),
    (116,8,"Supplements",2300,"2024-04-10"),
    (117,1,"Medicine",2600,"2024-04-15"),
    (118,2,"Medical Device",7800,"2024-04-18"),
    (119,3,"Supplements",2400,"2024-04-22"),
    (120,4,"Medicine",3300,"2024-04-25")
]
 
df_orders = spark.createDataFrame( orders, ["order_id","customer_id","product_category","sales_amount","order_date"] )
 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Part 1 — Data Understanding
## Task 1
Check:
- row count
- schema
- sample records

In [0]:
print("Row Count of orders: ", df_orders.count())
print("Row Count of customers: ", df_customers.count())

Row Count of orders:  20
Row Count of customers:  8


In [0]:
print(" Schema of data for Orders: ")
df_orders.printSchema()
print(" Schema of data for Customers: ")
df_customers.printSchema()

 Schema of data for Orders: 
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sales_amount: long (nullable = true)
 |-- order_date: string (nullable = true)

 Schema of data for Customers: 
root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)



In [0]:
print(" Printing 10 rows of data for orders: ")
df_orders.show(10)
print(" Printing 8 rows of data for customers: ")
df_customers.show()

 Printing 10 rows of data for orders: 
+--------+-----------+----------------+------------+----------+
|order_id|customer_id|product_category|sales_amount|order_date|
+--------+-----------+----------------+------------+----------+
|     101|          1|        Medicine|        2500|2024-01-05|
|     102|          1|     Supplements|        1800|2024-01-15|
|     103|          2|  Medical Device|        7500|2024-01-20|
|     104|          3|        Medicine|        3200|2024-02-01|
|     105|          4|     Supplements|        2200|2024-02-05|
|     106|          5|  Medical Device|        8500|2024-02-08|
|     107|          6|        Medicine|        2900|2024-02-12|
|     108|          7|     Supplements|        2100|2024-02-18|
|     109|          8|        Medicine|        3400|2024-03-01|
|     110|          2|        Medicine|        2800|2024-03-04|
+--------+-----------+----------------+------------+----------+
only showing top 10 rows
 Printing 8 rows of data for customers: 

## Task 2
Convert:
order_date
to proper date type.

In [0]:
df_orders = df_orders.withColumn( "order_date", to_date("order_date", "yyyy-MM-dd"))
df_orders.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sales_amount: long (nullable = true)
 |-- order_date: date (nullable = true)



# Part 2 — Joins
## Task 3
Join customers and orders.

In [0]:
df_customer_order = df_customers.join(df_orders, "customer_id", "inner")
df_customer_order.show()

+-----------+-------------+---------+--------+----------------+------------+----------+
|customer_id|customer_name|     city|order_id|product_category|sales_amount|order_date|
+-----------+-------------+---------+--------+----------------+------------+----------+
|          1|        Arjun|    Kochi|     101|        Medicine|        2500|2024-01-05|
|          1|        Arjun|    Kochi|     102|     Supplements|        1800|2024-01-15|
|          2|        Meera|  Chennai|     103|  Medical Device|        7500|2024-01-20|
|          3|        Rahul|Bangalore|     104|        Medicine|        3200|2024-02-01|
|          4|        Sneha|Hyderabad|     105|     Supplements|        2200|2024-02-05|
|          5|       Vikram|    Delhi|     106|  Medical Device|        8500|2024-02-08|
|          6|         Asha|    Kochi|     107|        Medicine|        2900|2024-02-12|
|          7|        Kiran|  Chennai|     108|     Supplements|        2100|2024-02-18|
|          8|         Neha|Banga

## Task 4
Verify:
row count before join
row count after join
Explain if they changed.

In [0]:
print(df_orders.count())
print(df_customer_order.count())

20
20


row count remained 20 before and after the join because each order is associated with exactly one valid customer, resulting in a one-to-many relationship where every order successfully finds a matching customer record.

# Part 3 — Transformations
## Task 5
Create:
sales_category
Rules:
- <3000 → Low
- 3000–7000 → Medium
- >7000→High

In [0]:
df_customer_order = df_customer_order.withColumn("sales_category", when(col("sales_amount") < 3000, "Low").when((col("sales_amount") >= 3000) & (col("sales_amount") < 7000), "Medium").otherwise("High"))
df_customer_order.show()

+-----------+-------------+---------+--------+----------------+------------+----------+--------------+
|customer_id|customer_name|     city|order_id|product_category|sales_amount|order_date|sales_category|
+-----------+-------------+---------+--------+----------------+------------+----------+--------------+
|          1|        Arjun|    Kochi|     101|        Medicine|        2500|2024-01-05|           Low|
|          1|        Arjun|    Kochi|     102|     Supplements|        1800|2024-01-15|           Low|
|          2|        Meera|  Chennai|     103|  Medical Device|        7500|2024-01-20|          High|
|          3|        Rahul|Bangalore|     104|        Medicine|        3200|2024-02-01|        Medium|
|          4|        Sneha|Hyderabad|     105|     Supplements|        2200|2024-02-05|           Low|
|          5|       Vikram|    Delhi|     106|  Medical Device|        8500|2024-02-08|          High|
|          6|         Asha|    Kochi|     107|        Medicine|        29


## Task 6
Extract:
month
from order_date.

In [0]:
df_customer_order = df_customer_order.withColumn("month", month(col("order_date")))
df_customer_order.show()

+-----------+-------------+---------+--------+----------------+------------+----------+--------------+-----+
|customer_id|customer_name|     city|order_id|product_category|sales_amount|order_date|sales_category|month|
+-----------+-------------+---------+--------+----------------+------------+----------+--------------+-----+
|          1|        Arjun|    Kochi|     101|        Medicine|        2500|2024-01-05|           Low|    1|
|          1|        Arjun|    Kochi|     102|     Supplements|        1800|2024-01-15|           Low|    1|
|          2|        Meera|  Chennai|     103|  Medical Device|        7500|2024-01-20|          High|    1|
|          3|        Rahul|Bangalore|     104|        Medicine|        3200|2024-02-01|        Medium|    2|
|          4|        Sneha|Hyderabad|     105|     Supplements|        2200|2024-02-05|           Low|    2|
|          5|       Vikram|    Delhi|     106|  Medical Device|        8500|2024-02-08|          High|    2|
|          6|      

# Part 4 — Aggregations
## Task 7
Find:
Total sales by city.

In [0]:
df_customer_order.groupBy("city").agg(sum("sales_amount").alias("total_sales")).show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|    Kochi|      12500|
|  Chennai|      28200|
|Hyderabad|       8600|
|Bangalore|      20300|
|    Delhi|      11000|
+---------+-----------+



## Task 8
Find:
Total sales by product_category.

In [0]:
df_customer_order.groupBy("product_category").agg(sum("sales_amount").alias("total_sales")).show()

+----------------+-----------+
|product_category|total_sales|
+----------------+-----------+
|        Medicine|      26500|
|     Supplements|      13300|
|  Medical Device|      40800|
+----------------+-----------+



## Task 9
Find:
Average sales by city.

In [0]:
df_customer_order.groupBy("city").agg(round(avg("sales_amount"),2).alias("avg_sales")).show()

+---------+---------+
|     city|avg_sales|
+---------+---------+
|    Kochi|   2500.0|
|  Chennai|   5640.0|
|Hyderabad|  2866.67|
|Bangalore|   4060.0|
|    Delhi|   5500.0|
+---------+---------+



## Task 10
Find:
Number of orders by city.

In [0]:
df_customer_order.groupBy("city").agg(count("order_id").alias("count_sales")).show()

+---------+-----------+
|     city|count_sales|
+---------+-----------+
|    Kochi|          5|
|  Chennai|          5|
|Hyderabad|          3|
|Bangalore|          5|
|    Delhi|          2|
+---------+-----------+



# Part 5 — Pivot
## Task 11
Create pivot:
Rows:
city
Columns:
product_category
Values:
total sales

In [0]:
pivot_table = df_customer_order.groupBy("city").pivot("product_category").agg(sum("sales_amount"))
pivot_table.show()

+---------+--------------+--------+-----------+
|     city|Medical Device|Medicine|Supplements|
+---------+--------------+--------+-----------+
|    Kochi|          NULL|   10700|       1800|
|Hyderabad|          NULL|    6400|       2200|
|Bangalore|          9000|    6600|       4700|
|  Chennai|         23300|    2800|       2100|
|    Delhi|          8500|    NULL|       2500|
+---------+--------------+--------+-----------+



## Task 12
Replace pivot nulls with 0.

In [0]:
pivot_table = pivot_table.fillna(0)
pivot_table.show()

+---------+--------------+--------+-----------+
|     city|Medical Device|Medicine|Supplements|
+---------+--------------+--------+-----------+
|    Kochi|             0|   10700|       1800|
|Hyderabad|             0|    6400|       2200|
|Bangalore|          9000|    6600|       4700|
|  Chennai|         23300|    2800|       2100|
|    Delhi|          8500|       0|       2500|
+---------+--------------+--------+-----------+



# Part 6 — Window Functions
## Task 13
Rank customers based on sales amount within each city.

In [0]:
customer_sales = df_customer_order.groupBy("customer_id", "customer_name", "city").agg(sum("sales_amount").alias("total_sales"))
# customer_sales.show()
cus_rank = customer_sales.withColumn("rank", dense_rank().over(Window.partitionBy("city").orderBy(desc("total_sales"))))
cus_rank.show()

+-----------+-------------+---------+-----------+----+
|customer_id|customer_name|     city|total_sales|rank|
+-----------+-------------+---------+-----------+----+
|          3|        Rahul|Bangalore|      14600|   1|
|          8|         Neha|Bangalore|       5700|   2|
|          2|        Meera|  Chennai|      18100|   1|
|          7|        Kiran|  Chennai|      10100|   2|
|          5|       Vikram|    Delhi|      11000|   1|
|          4|        Sneha|Hyderabad|       8600|   1|
|          1|        Arjun|    Kochi|       6900|   1|
|          6|         Asha|    Kochi|       5600|   2|
+-----------+-------------+---------+-----------+----+



## Task 14
Find top customer in each city.

In [0]:
cus_rank.orderBy(desc("rank")).show()

+-----------+-------------+---------+-----------+----+
|customer_id|customer_name|     city|total_sales|rank|
+-----------+-------------+---------+-----------+----+
|          8|         Neha|Bangalore|       5700|   2|
|          7|        Kiran|  Chennai|      10100|   2|
|          6|         Asha|    Kochi|       5600|   2|
|          3|        Rahul|Bangalore|      14600|   1|
|          2|        Meera|  Chennai|      18100|   1|
|          5|       Vikram|    Delhi|      11000|   1|
|          4|        Sneha|Hyderabad|       8600|   1|
|          1|        Arjun|    Kochi|       6900|   1|
+-----------+-------------+---------+-----------+----+



## Task 15
Calculate previous sales amount for each customer.
(Hint: lag)

In [0]:
df_customer_order = df_customer_order.withColumn("previous_sale", lag("sales_amount", 1).over(Window.partitionBy("customer_id").orderBy(desc("sales_amount"))))
df_customer_order.show()

+-----------+-------------+---------+--------+----------------+------------+----------+--------------+-----+-------------+
|customer_id|customer_name|     city|order_id|product_category|sales_amount|order_date|sales_category|month|previous_sale|
+-----------+-------------+---------+--------+----------------+------------+----------+--------------+-----+-------------+
|          1|        Arjun|    Kochi|     117|        Medicine|        2600|2024-04-15|           Low|    4|         NULL|
|          1|        Arjun|    Kochi|     101|        Medicine|        2500|2024-01-05|           Low|    1|         2600|
|          1|        Arjun|    Kochi|     102|     Supplements|        1800|2024-01-15|           Low|    1|         2500|
|          2|        Meera|  Chennai|     118|  Medical Device|        7800|2024-04-18|          High|    4|         NULL|
|          2|        Meera|  Chennai|     103|  Medical Device|        7500|2024-01-20|          High|    1|         7800|
|          2|   

## Task 16
Calculate sales difference.

In [0]:
df_customer_order.withColumn("sales_difference", col("sales_amount") - col("previous_sale")).display()

customer_id,customer_name,city,order_id,product_category,sales_amount,order_date,sales_category,month,previous_sale,sales_difference
1,Arjun,Kochi,117,Medicine,2600,2024-04-15,Low,4,null,null
1,Arjun,Kochi,101,Medicine,2500,2024-01-05,Low,1,2600,-100
1,Arjun,Kochi,102,Supplements,1800,2024-01-15,Low,1,2500,-700
2,Meera,Chennai,118,Medical Device,7800,2024-04-18,High,4,null,null
2,Meera,Chennai,103,Medical Device,7500,2024-01-20,High,1,7800,-300
2,Meera,Chennai,110,Medicine,2800,2024-03-04,Low,3,7500,-4700
3,Rahul,Bangalore,111,Medical Device,9000,2024-03-10,High,3,null,null
3,Rahul,Bangalore,104,Medicine,3200,2024-02-01,Medium,2,9000,-5800
3,Rahul,Bangalore,119,Supplements,2400,2024-04-22,Low,4,3200,-800
4,Sneha,Hyderabad,120,Medicine,3300,2024-04-25,Medium,4,null,null


# Part 7 — SQL
## Task 17
Create temp view.

In [0]:
df_customer_order.createOrReplaceTempView("cus_order")

## Task 18
Write SQL to find:
Total sales by city.

In [0]:
spark.sql(""" SELECT city, SUM(sales_amount) AS Total_sales FROM cus_order GROUP BY CITY""").show()

+---------+-----------+
|     city|Total_sales|
+---------+-----------+
|    Kochi|      12500|
|  Chennai|      28200|
|Hyderabad|       8600|
|Bangalore|      20300|
|    Delhi|      11000|
+---------+-----------+



## Task 19
Write SQL to find:
Top city by sales.

In [0]:
spark.sql(""" SELECT city, SUM(sales_amount) AS Total_sales FROM cus_order GROUP BY city ORDER BY Total_sales DESC LIMIT 1 """).show()

+-------+-----------+
|   city|Total_sales|
+-------+-----------+
|Chennai|      28200|
+-------+-----------+



# Part 8 — Business Thinking
## Task 20
Which city is performing best?

In [0]:
spark.sql(""" SELECT city, SUM(sales_amount) AS Total_sales FROM cus_order GROUP BY city ORDER BY Total_sales DESC LIMIT 1 """).show()

+-------+-----------+
|   city|Total_sales|
+-------+-----------+
|Chennai|      28200|
+-------+-----------+



## Task 21
Which product category is most valuable?

In [0]:
spark.sql(""" SELECT product_category, SUM(sales_amount) AS Total_sales FROM cus_order GROUP BY product_category ORDER BY Total_sales DESC LIMIT 1 """).show()

+----------------+-----------+
|product_category|Total_sales|
+----------------+-----------+
|  Medical Device|      40800|
+----------------+-----------+



## Task 22
Which customers should be targeted for loyalty programs?

In [0]:
spark.sql(""" SELECT customer_id, customer_name, SUM(sales_amount) AS Total_sales FROM cus_order GROUP BY customer_id, customer_name ORDER BY Total_sales DESC LIMIT 1 """).show()

+-----------+-------------+-----------+
|customer_id|customer_name|Total_sales|
+-----------+-------------+-----------+
|          2|        Meera|      18100|
+-----------+-------------+-----------+



## Task 23
Provide 5 business insights.
 


In [0]:
print("The city that has highest business is :", df_customer_order.groupBy("city").agg(sum("sales_amount").alias("total_sales")).orderBy(desc("total_sales")).collect()[0][0])
print("---------------------------------------------------")
print("The product category that has highest business is :", df_customer_order.groupBy("product_category").agg(sum("sales_amount").alias("total_sales")).orderBy(desc("total_sales")).collect()[0][0])
print("---------------------------------------------------")
print("The customer that has highest business is :", df_customer_order.groupBy("customer_id", "customer_name").agg(sum("sales_amount").alias("total_sales")).orderBy(desc("total_sales")).collect()[0][1])
print("---------------------------------------------------")
print("Most Repeated Customers:")
df_customer_order.groupBy("customer_name").agg(count("order_id").alias("order_count")).orderBy(desc("order_count")).show() 
print("---------------------------------------------------")
print("best city to invest in the fastest-growing market ", df_customer_order.withColumn("growth_rate", col("sales_amount") - col("previous_sale")/col("previous_sale")).groupBy("city").agg(sum("growth_rate").alias("growth_rate")).orderBy(desc("growth_rate")).collect()[0][0])
print("---------------------------------------------------")
print("best city to invest in the most consistent market ", df_customer_order.groupBy("city").agg(stddev("sales_amount").alias("stddev_sales")).orderBy("stddev_sales").collect()[0][0], "smallest fluctuation.")
print("---------------------------------------------------")
print("Which City Needs More Inventory? ", df_customer_order.groupBy("city").agg(sum("sales_amount").alias("revenue"), count("order_id").alias("orders")).orderBy(desc("revenue"), desc("orders")).collect()[0][0])

The city that has highest business is : Chennai
---------------------------------------------------
The product category that has highest business is : Medical Device
---------------------------------------------------
The customer that has highest business is : Meera
---------------------------------------------------
Most Repeated Customers:
+-------------+-----------+
|customer_name|order_count|
+-------------+-----------+
|        Arjun|          3|
|        Rahul|          3|
|        Sneha|          3|
|        Meera|          3|
|       Vikram|          2|
|         Asha|          2|
|        Kiran|          2|
|         Neha|          2|
+-------------+-----------+

---------------------------------------------------
best city to invest in the fastest-growing market  Chennai
---------------------------------------------------
best city to invest in the most consistent market  Kochi smallest fluctuation.
---------------------------------------------------
Which City Needs More I